<a href="https://colab.research.google.com/github/DivyamAwasthy/Paper-Insight/blob/main/paper_insight_stage1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q sentence-transformers feedparser

import feedparser
import numpy as np
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity


def fetch_arxiv(query: str = "cat:cs.LG", max_results: int = 30):
    """Fetch (title, abstract) pairs from arXiv for a given search query."""
    url = (
        "http://export.arxiv.org/api/query?"
        f"search_query={query}&start=0&max_results={max_results}"
        "&sortBy=submittedDate&sortOrder=descending"
    )
    feed = feedparser.parse(url)
    papers = []
    for entry in feed.entries:
        papers.append({
            "title": entry.title.strip().replace("\n", " "),
            "abstract": entry.summary.strip().replace("\n", " "),
        })
    return papers


MODEL_NAME = "all-MiniLM-L6-v2"


def embed_texts(model, texts):
    return model.encode(
        texts,
        normalize_embeddings=True,
        show_progress_bar=True,
    )


def search(model, query, corpus_embeddings, papers, top_k=5):
    query_emb = model.encode([query], normalize_embeddings=True)
    sims = cosine_similarity(query_emb, corpus_embeddings)[0]
    ranked = np.argsort(sims)[::-1][:top_k]
    results = []
    for idx in ranked:
        results.append({
            "score": float(sims[idx]),
            "title": papers[idx]["title"],
            "abstract": papers[idx]["abstract"],
        })
    return results

print("Functions loaded: fetch_arxiv, embed_texts, search")

In [ ]:
papers = fetch_arxiv(query="all:graph+neural+network", max_results=100)
print(f"got {len(papers)} papers")

model = SentenceTransformer(MODEL_NAME)
corpus_embeddings = embed_texts(model, [p["abstract"] for p in papers])
print("embeddings shape:", corpus_embeddings.shape)   # expect (100, 384)

In [ ]:
hits = search(model, "graph neural networks for molecules", corpus_embeddings, papers, top_k=5)
for i, h in enumerate(hits, 1):
    print(f"\n[{i}] score={h['score']:.3f}")
    print("   ", h["title"])
    print("   ", h["abstract"][:200], "...")

In [ ]:
papers = fetch_arxiv(query="all:graph+neural+network", max_results=100)
print(f"got {len(papers)} papers")

model = SentenceTransformer(MODEL_NAME)
corpus_embeddings = embed_texts(model, [p["abstract"] for p in papers])
print("embeddings shape:", corpus_embeddings.shape)   # expect (30, 384)